# Datathon UCU-ITBA - Carga, exportacion y analisis general

**Objetivo de este notebook:** abrir los cuatro parquet provistos, dejarlos disponibles en CSV
y producir un diagnostico general de cada tabla (estructura, nulos, duplicados, dominios de
valores e incoherencias logicas).

No limpia nada todavia: solo mide. Las decisiones de limpieza se toman despues, con esta
evidencia sobre la mesa, y quedan registradas en el log de decisiones del equipo.

Tablas: `personas`, `laboral_ingresos`, `informacion_crediticia`, `geografia_hogar`.
Claves comunes: `id_persona` + `periodo`.

## 1. Configuracion y carga

In [57]:
import os
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

BASE = os.getcwd()   # el notebook debe abrirse desde la carpeta que contiene los .parquet
DIR_CSV = os.path.join(BASE, "csv")
os.makedirs(DIR_CSV, exist_ok=True)

TABLAS = ["personas", "laboral_ingresos", "informacion_crediticia", "geografia_hogar"]
print("Carpeta de trabajo:", BASE)

Carpeta de trabajo: c:\Users\nicok\OneDrive\NICO\ITBA\Año 3\C1\Dathaton


In [58]:
dfs = {}
for t in TABLAS:
    ruta = os.path.join(BASE, f"{t}.parquet")
    dfs[t] = pd.read_parquet(ruta)
    print(f"{t:<26} {dfs[t].shape[0]:>8,} filas x {dfs[t].shape[1]:>2} columnas")

personas   = dfs["personas"]
laboral    = dfs["laboral_ingresos"]
crediticia = dfs["informacion_crediticia"]
geografia  = dfs["geografia_hogar"]

personas                    866,780 filas x  4 columnas
laboral_ingresos            897,041 filas x  8 columnas
informacion_crediticia      866,757 filas x 13 columnas
geografia_hogar             876,271 filas x  9 columnas


## 2. Exportacion a CSV

Se exporta cada tabla tal cual viene, sin ninguna transformacion, a la subcarpeta `csv/`.
Separador `,`, codificacion `utf-8-sig` para que Excel respete los acentos.

In [59]:
for t, df in dfs.items():
    destino = os.path.join(DIR_CSV, f"{t}.csv")
    origen = os.path.join(BASE, f"{t}.parquet")
    if os.path.exists(destino) and os.path.getmtime(destino) > os.path.getmtime(origen):
        mb = os.path.getsize(destino) / 1024**2
        print(f"{t:<26} ya existe, se omite      ({mb:,.1f} MB)")
        continue
    df.to_csv(destino, index=False, encoding="utf-8-sig")
    mb = os.path.getsize(destino) / 1024**2
    print(f"{t:<26} -> csv/{t}.csv           ({mb:,.1f} MB)")


personas                   ya existe, se omite      (18.2 MB)
laboral_ingresos           ya existe, se omite      (39.6 MB)
informacion_crediticia     ya existe, se omite      (81.7 MB)
geografia_hogar            ya existe, se omite      (32.8 MB)


## 3. Diagnostico general por tabla

Para cada tabla: tipo de dato, porcentaje de nulos, cardinalidad y una muestra de los
valores mas frecuentes. Sirve para detectar de un vistazo columnas inutilizables
(demasiados nulos), columnas sin varianza y categorias mal codificadas.

In [60]:
def perfil(df, nombre, max_valores=4):
    filas = []
    for col in df.columns:
        s = df[col]
        nulos = s.isna().mean() * 100
        top = s.dropna().astype(str).value_counts().head(max_valores)
        muestra = " | ".join(f"{k} ({v:,})" for k, v in top.items())
        filas.append({
            "columna": col,
            "tipo": str(s.dtype),
            "nulos_%": round(nulos, 2),
            "unicos": s.nunique(dropna=True),
            "valores_frecuentes": muestra[:110],
        })
    out = pd.DataFrame(filas)
    print(f"\n{'='*100}\n{nombre.upper()}  -  {df.shape[0]:,} filas x {df.shape[1]} columnas\n{'='*100}")
    return out

def resumen_claves(df, nombre):
    n_filas = len(df)
    n_pers = df["id_persona"].nunique()
    dups = df.duplicated(["id_persona", "periodo"]).sum() if "periodo" in df.columns else np.nan
    dups_full = df.duplicated().sum()
    print(f"{nombre:<26} filas={n_filas:>8,}  personas={n_pers:>8,}  "
          f"dup(id,periodo)={dups:>7,}  dup(fila completa)={dups_full:>7,}")

In [61]:
for t in TABLAS:
    display(perfil(dfs[t], t))


PERSONAS  -  866,780 filas x 4 columnas


,columna,tipo,nulos_%,unicos,valores_frecuentes
0,id_persona,str,0.00,674546,b5034cc5 (3) | b75e3e9d (3) | b76d7b4f (3) | c...
1,periodo,str,0.00,3,"2026-06 (288,981) | 2025-12 (288,900) | 2025-0..."
2,genero,str,0.00,3,"F (436,901) | M (420,052) | X (9,827)"
3,edad,int64,0.00,112,"35 (24,515) | 34 (21,588) | 45 (20,926) | 43 (..."



LABORAL_INGRESOS  -  897,041 filas x 8 columnas


,columna,tipo,nulos_%,unicos,valores_frecuentes
0,id_persona,str,0.00,674538,bf0c11a7 (6) | b44643d2 (6) | dec13720 (6) | 8...
1,periodo,str,0.00,3,"2026-06 (299,133) | 2025-06 (298,973) | 2025-1..."
2,categoria_ingreso,str,20.89,5,"MEDIO-BAJO (257,280) | MEDIO (176,499) | BAJO ..."
3,es_relacion_dependencia,str,26.23,2,"FALSE (534,130) | TRUE (127,592)"
4,es_monotributo_o_autonomo,str,26.23,2,"FALSE (541,349) | TRUE (120,373)"
5,es_pasivo,str,26.23,2,"FALSE (577,182) | TRUE (84,540)"
6,tiene_obra_social_o_prepaga,str,26.17,2,"TRUE (387,080) | FALSE (275,239)"
7,beneficiario_plan_social_12m,object,26.17,2,"False (661,376) | True (943)"



INFORMACION_CREDITICIA  -  866,757 filas x 13 columnas


,columna,tipo,nulos_%,unicos,valores_frecuentes
0,id_persona,str,0.00,674536,3f867926 (3) | 58566c5a (3) | a2512f81 (3) | 8...
1,periodo,str,8.10,3,"2026-06 (265,539) | 2025-06 (265,503) | 2025-1..."
2,bancarizacion,str,0.00,2,"HIT (633,537) | THIN (233,220)"
3,segmento_comportamiento_retail,str,0.00,4,"Frecuente (217,274) | Habitual (216,648) | Pre..."
4,score_riesgo,str,7.10,10,"SCORE_MEDIO_ALTO (169,777) | SCORE_ALTO (142,4..."
5,score_fraude,str,0.00,4,"MUY BAJO (736,355) | BAJO (86,991) | MEDIO (30..."
6,cantidad_solicitudes_financiamiento_6m,str,0.00,3,"NO SOLICITO (633,507) | ENTRE 1 Y 3 (203,018) ..."
7,cantidad_solicitudes_financiamiento_12m,str,0.00,3,"NO SOLICITO (498,489) | ENTRE 1 Y 3 (279,606) ..."
8,cantidad_solicitudes_financiamiento_24m,str,0.00,3,"NO SOLICITO (349,727) | ENTRE 1 Y 3 (330,114) ..."
9,abrio_linea_nueva_ult_12m,str,0.00,2,"FALSE (746,220) | TRUE (120,537)"



GEOGRAFIA_HOGAR  -  876,271 filas x 9 columnas


,columna,tipo,nulos_%,unicos,valores_frecuentes
0,id_persona,str,0.00,674546,914804c0 (5) | c5c7d9ea (5) | 756a4c57 (5) | 9...
1,periodo,str,0.00,3,"2026-06 (292,208) | 2025-06 (292,097) | 2025-1..."
2,cp,str,0.00,47,"5000 (113,593) | 2000 (92,597) | 7600 (63,591)..."
3,distancia_estimada_polo_comercial_km,float64,0.00,501,"2.63 (1,860) | 0.44 (1,860) | 1.78 (1,857) | 2..."
4,edad_promedio_hogar,float64,71.10,1016,"41.0 (2,632) | 38.0 (2,620) | 39.0 (2,554) | 4..."
5,cantidad_personas_hogar,float64,71.10,13,"2.0 (138,976) | 3.0 (47,257) | 4.0 (38,023) | ..."
6,cantidad_menores_hogar,float64,71.10,8,"0.0 (199,354) | 1.0 (36,910) | 2.0 (13,902) | ..."
7,score_riesgo_promedio_hogar,str,71.10,13,"MEDIO-ALTO (89,132) | ALTO (56,288) | MEDIO (4..."
8,categoria_ingreso_lider_hogar,str,71.10,5,"BAJO (91,732) | MEDIO (68,781) | MEDIO-BAJO (5..."


In [62]:
print("Integridad de claves\n" + "-"*80)
for t in TABLAS:
    resumen_claves(dfs[t], t)

Integridad de claves
--------------------------------------------------------------------------------
personas                   filas= 866,780  personas= 674,546  dup(id,periodo)=      0  dup(fila completa)=      0
laboral_ingresos           filas= 897,041  personas= 674,538  dup(id,periodo)= 30,276  dup(fila completa)= 30,268
informacion_crediticia     filas= 866,757  personas= 674,536  dup(id,periodo)=  1,760  dup(fila completa)=     28
geografia_hogar            filas= 876,271  personas= 674,546  dup(id,periodo)=  9,491  dup(fila completa)=      1


## 4. Cobertura temporal

El dataset trae tres cortes. La pregunta clave es si se trata de un panel (las mismas personas
observadas en el tiempo) o de un corte transversal apilado, porque eso condiciona todo el
analisis posterior.

In [63]:
print(personas["periodo"].value_counts().sort_index().to_string(), "\n")

periodos_por_persona = personas.groupby("id_persona")["periodo"].nunique().value_counts().sort_index()
tabla = periodos_por_persona.rename("personas").to_frame()
tabla["%"] = (tabla["personas"] / tabla["personas"].sum() * 100).round(1)
tabla.index.name = "cantidad de periodos observados"
display(tabla)

periodo
2025-06    288899
2025-12    288900
2026-06    288981 



,personas,%
cantidad de periodos observados,,
1,572413,84.90
2,12032,1.80
3,90101,13.40


## 5. Chequeos de calidad, tabla por tabla

Cada bloque mide un problema concreto. Los resultados se anotan en el log de decisiones
junto con el tratamiento elegido.

### 5.1 `personas`

In [64]:
print("Genero:")
print(personas["genero"].value_counts(dropna=False).to_string(), "\n")

print("Edad:")
print(personas["edad"].describe().to_string(), "\n")
print("edad < 18 :", (personas["edad"] < 18).sum())
print("edad > 90 :", (personas["edad"] > 90).sum())
print("edad > 100:", (personas["edad"] > 100).sum(), "\n")

# Consistencia longitudinal: el genero no deberia cambiar y la edad no deberia bajar
gen_inconsistente = personas.groupby("id_persona")["genero"].nunique()
print("personas con genero distinto entre periodos:", (gen_inconsistente > 1).sum())

piv = personas.pivot_table(index="id_persona", columns="periodo", values="edad", aggfunc="max")
if {"2025-06", "2026-06"}.issubset(piv.columns):
    amb = piv.dropna(subset=["2025-06", "2026-06"])
    print("personas cuya edad BAJA entre 2025-06 y 2026-06:",
          int((amb["2026-06"] < amb["2025-06"]).sum()), "de", len(amb))

Genero:
genero
F    436901
M    420052
X      9827 

Edad:
count   866,780.00
mean         46.09
std          17.94
min          18.00
25%          33.00
50%          43.00
75%          57.00
max         129.00 

edad < 18 : 0
edad > 90 : 13436
edad > 100: 6062 

personas con genero distinto entre periodos: 347
personas cuya edad BAJA entre 2025-06 y 2026-06: 462 de 93332


### 5.2 `laboral_ingresos`

In [65]:
cols_flag = ["es_relacion_dependencia", "es_monotributo_o_autonomo", "es_pasivo"]

print("categoria_ingreso:")
print(laboral["categoria_ingreso"].value_counts(dropna=False).to_string(), "\n")

for c in cols_flag + ["tiene_obra_social_o_prepaga", "beneficiario_plan_social_12m"]:
    print(f"{c}:")
    print(laboral[c].astype(str).value_counts(dropna=False).to_string(), "\n")

# Incoherencia: una persona no puede estar en dos situaciones laborales a la vez
sub = laboral.dropna(subset=cols_flag)
n_true = (sub[cols_flag].astype(str) == "TRUE").sum(axis=1)
print("Cantidad de flags laborales simultaneamente TRUE:")
print(n_true.value_counts().sort_index().to_string())
print("\nregistros con mas de una situacion laboral:", int((n_true > 1).sum()))

categoria_ingreso:
categoria_ingreso
MEDIO-BAJO    257280
NaN           187381
MEDIO         176499
BAJO          145380
MEDIO-ALTO     78449
ALTO           52052 

es_relacion_dependencia:
es_relacion_dependencia
FALSE    534130
NaN      235319
TRUE     127592 

es_monotributo_o_autonomo:
es_monotributo_o_autonomo
FALSE    541349
NaN      235319
TRUE     120373 

es_pasivo:
es_pasivo
FALSE    577182
NaN      235319
TRUE      84540 

tiene_obra_social_o_prepaga:
tiene_obra_social_o_prepaga
TRUE     387080
FALSE    275239
NaN      234722 

beneficiario_plan_social_12m:
beneficiario_plan_social_12m
False    661376
NaN      234722
True        943 

Cantidad de flags laborales simultaneamente TRUE:
0    357641
1    275902
2     27934
3       245

registros con mas de una situacion laboral: 28179


In [87]:
# Los duplicados por (id_persona, periodo): son copias identicas o registros contradictorios?
dup = laboral[laboral.duplicated(["id_persona", "periodo"], keep=False)]
print("filas involucradas en duplicados:", f"{len(dup):,}")
contradictorios = dup.groupby(["id_persona", "periodo"])["categoria_ingreso"].nunique(dropna=False)
print("pares duplicados totales      :", f"{len(contradictorios):,}")
print("pares con ingreso contradictorio:", int((contradictorios > 1).sum()))
display(dup.sort_values(["id_persona", "periodo"]).head(6))

filas involucradas en duplicados: 60,551
pares duplicados totales      : 30,275
pares con ingreso contradictorio: 5


,id_persona,periodo,categoria_ingreso,es_relacion_dependencia,es_monotributo_o_autonomo,es_pasivo,tiene_obra_social_o_prepaga,beneficiario_plan_social_12m
117074,000158c1,2025-12,MEDIO-BAJO,NaN,NaN,NaN,NaN,None
154279,000158c1,2025-12,MEDIO-BAJO,NaN,NaN,NaN,NaN,None
804002,00017cdb,2026-06,MEDIO-BAJO,FALSE,FALSE,FALSE,TRUE,False
811779,00017cdb,2026-06,MEDIO-BAJO,FALSE,FALSE,FALSE,TRUE,False
519378,00019bb8,2025-06,MEDIO-ALTO,FALSE,FALSE,FALSE,TRUE,False
547823,00019bb8,2025-06,MEDIO-ALTO,FALSE,FALSE,FALSE,TRUE,False


### 5.3 `informacion_crediticia`

In [67]:
for c in ["bancarizacion", "segmento_comportamiento_retail", "score_riesgo", "score_fraude"]:
    print(f"{c}:")
    print(crediticia[c].astype(str).value_counts(dropna=False).to_string(), "\n")

bancarizacion:
bancarizacion
HIT     633537
THIN    233220 

segmento_comportamiento_retail:
segmento_comportamiento_retail
Frecuente             217274
Habitual              216648
Premium/Heavy User    216609
Ocasional             216226 

score_riesgo:
score_riesgo
SCORE_MEDIO_ALTO    169777
SCORE_ALTO          142479
SCORE_MEDIO          98156
MEDIO_ALTO           80137
SCORE_BAJO           74808
NaN                  61532
ALTO                 60447
SCORE_MEDIO_BAJO     51040
MEDIO                50296
BAJO                 50224
MEDIO_BAJO           27861 

score_fraude:
score_fraude
MUY BAJO    736355
BAJO         86991
MEDIO        30430
ALTO         12981 



In [68]:
# score_riesgo llega con dos codificaciones conviviendo (SCORE_ALTO vs ALTO).
# Si no se unifican, cada nivel de riesgo queda partido en dos.
sr = crediticia["score_riesgo"].dropna().astype(str)
con_prefijo = sr.str.startswith("SCORE_")
print("valores con prefijo SCORE_:", f"{con_prefijo.sum():,}")
print("valores sin prefijo       :", f"{(~con_prefijo).sum():,}\n")
print("niveles al unificar el prefijo:")
print(sr.str.replace("^SCORE_", "", regex=True).value_counts().to_string())

valores con prefijo SCORE_: 536,260
valores sin prefijo       : 268,965

niveles al unificar el prefijo:
score_riesgo
MEDIO_ALTO    249914
ALTO          202926
MEDIO         148452
BAJO          125032
MEDIO_BAJO     78901


In [69]:
cols_lineas = ["cantidad_lineas_credito_activas_12m",
               "cantidad_lineas_credito_activas_24m",
               "cantidad_lineas_credito_activas_36m"]

print(crediticia[cols_lineas].describe().to_string(), "\n")

print("24m y 36m son identicas en todas las filas:",
      bool((crediticia[cols_lineas[1]] == crediticia[cols_lineas[2]]).all()))
print("filas con 12m > 24m (imposible):",
      f"{int((crediticia[cols_lineas[0]] > crediticia[cols_lineas[1]]).sum()):,}")
print("\npercentiles altos de lineas_12m:")
print(crediticia[cols_lineas[0]].quantile([.9, .95, .99, .999, 1]).to_string())
print("\nfilas sin periodo:", f"{crediticia['periodo'].isna().sum():,}",
      f"({crediticia['periodo'].isna().mean()*100:.1f}%)")

       cantidad_lineas_credito_activas_12m  cantidad_lineas_credito_activas_24m  cantidad_lineas_credito_activas_36m
count                           866,757.00                           866,757.00                           866,757.00
mean                                  2.23                                 2.24                                 2.24
std                                   3.18                                 2.72                                 2.72
min                                   0.00                                 0.00                                 0.00
25%                                   0.00                                 0.00                                 0.00
50%                                   1.00                                 2.00                                 2.00
75%                                   3.00                                 3.00                                 3.00
max                                 384.00                      

In [70]:
# Coherencia de la escalera de solicitudes: 6m <= 12m <= 24m
orden = {"NO SOLICITO": 0, "ENTRE 1 Y 3": 1, "4 O MAS": 2, "4 O MÁS": 2}
sol = crediticia[["cantidad_solicitudes_financiamiento_6m",
                  "cantidad_solicitudes_financiamiento_12m",
                  "cantidad_solicitudes_financiamiento_24m"]].apply(lambda s: s.map(orden))
print("filas con 6m > 12m :", f"{int((sol.iloc[:,0] > sol.iloc[:,1]).sum()):,}")
print("filas con 12m > 24m:", f"{int((sol.iloc[:,1] > sol.iloc[:,2]).sum()):,}")

filas con 6m > 12m : 0
filas con 12m > 24m: 0


### 5.4 `geografia_hogar`

In [71]:
cols_hogar = ["edad_promedio_hogar", "cantidad_personas_hogar", "cantidad_menores_hogar",
              "score_riesgo_promedio_hogar", "categoria_ingreso_lider_hogar"]

print("codigos postales distintos:", geografia["cp"].nunique(), "\n")
print("distancia al polo comercial (km):")
print(geografia["distancia_estimada_polo_comercial_km"].describe().to_string(), "\n")
print("nulos en variables de hogar:")
print((geografia[cols_hogar].isna().mean() * 100).round(1).to_string())

codigos postales distintos: 47 

distancia al polo comercial (km):
count   876,271.00
mean          2.50
std           1.44
min           0.00
25%           1.25
50%           2.50
75%           3.75
max           5.00 

nulos en variables de hogar:
edad_promedio_hogar             71.10
cantidad_personas_hogar         71.10
cantidad_menores_hogar          71.10
score_riesgo_promedio_hogar     71.10
categoria_ingreso_lider_hogar   71.10


In [72]:
# El bloque de hogar, falta en todo o en nada?
faltan = geografia[cols_hogar].isna().sum(axis=1)
print("columnas de hogar faltantes por fila:")
print(faltan.value_counts().sort_index().to_string(), "\n")

# La falta de dato se reparte pareja entre CPs? Si no, hay sesgo geografico.
cob = (geografia.assign(tiene_hogar=geografia["cantidad_personas_hogar"].notna())
                .groupby("cp")
                .agg(registros=("id_persona", "size"),
                     cobertura_hogar_pct=("tiene_hogar", lambda s: round(s.mean()*100, 1)))
                .sort_values("registros", ascending=False))
display(cob.head(15))
print("\ncobertura minima / maxima entre CPs:",
      cob["cobertura_hogar_pct"].min(), "/", cob["cobertura_hogar_pct"].max())

columnas de hogar faltantes por fila:
0    253259
5    623012 



,registros,cobertura_hogar_pct
cp,,
5000,113593,27.80
2000,92597,29.30
7600,63591,34.60
4000,52817,23.20
1900,49300,28.90
4400,47695,23.20
3000,34792,30.10
1425,32505,26.70
3400,29522,22.50



cobertura minima / maxima entre CPs: 22.5 / 44.1


In [73]:
# Tamano de cada plaza, medido en personas unicas. Es el universo de decision del datathon.
plazas = (geografia.drop_duplicates("id_persona")
                   .groupby("cp")
                   .agg(personas=("id_persona", "nunique"),
                        dist_media_km=("distancia_estimada_polo_comercial_km", "mean"))
                   .sort_values("personas", ascending=False))
plazas["%_muestra"] = (plazas["personas"] / plazas["personas"].sum() * 100).round(2)
plazas["dist_media_km"] = plazas["dist_media_km"].round(2)
print("total de personas unicas:", f"{plazas['personas'].sum():,}")
display(plazas.head(20))

total de personas unicas: 674,546


,personas,dist_media_km,%_muestra
cp,,,
5000,86727,2.50,12.86
2000,69076,2.51,10.24
7600,46971,2.50,6.96
4000,39288,2.50,5.82
1900,36631,2.49,5.43
4400,35652,2.51,5.29
3000,26047,2.50,3.86
1425,24257,2.49,3.60
3400,22121,2.50,3.28


## 6. Integridad entre tablas

Todas las tablas deberian describir al mismo conjunto de personas. Cualquier diferencia
obliga a decidir si el cruce final es un `inner join` (se pierden personas) o un `left join`
desde `personas` (aparecen nulos nuevos).

In [74]:
ids = {t: set(dfs[t]["id_persona"].unique()) for t in TABLAS}
base = ids["personas"]

print(f"{'tabla':<26} {'personas':>10} {'faltan vs personas':>20} {'sobran vs personas':>20}")
for t in TABLAS:
    print(f"{t:<26} {len(ids[t]):>10,} {len(base - ids[t]):>20,} {len(ids[t] - base):>20,}")

comunes = set.intersection(*ids.values())
print(f"\npersonas presentes en las 4 tablas: {len(comunes):,} "
      f"({len(comunes)/len(base)*100:.1f}% de personas)")

tabla                        personas   faltan vs personas   sobran vs personas
personas                      674,546                    0                    0
laboral_ingresos              674,538                    8                    0
informacion_crediticia        674,536                   10                    0
geografia_hogar               674,546                    0                    0

personas presentes en las 4 tablas: 674,536 (100.0% de personas)


## 6. Senal vs ruido: que variables sirven realmente

Antes de construir cualquier indice hay que separar las variables que discriminan entre
plazas de las que son ruido. Una variable con distribucion identica en todas las plazas no
aporta nada al ranking, por mas que parezca relevante por su nombre.

In [75]:
# Ultimo registro por persona: base de trabajo para todo el analisis por plaza.
ORDEN = {"2025-06": 0, "2025-12": 1, "2026-06": 2}

def ultimo_registro(df):
    d = df.copy()
    d["_o"] = d["periodo"].map(ORDEN).fillna(-1)
    return d.sort_values("_o").drop_duplicates("id_persona", keep="last").drop(columns="_o")

P = ultimo_registro(personas)
L = ultimo_registro(laboral)
C = ultimo_registro(crediticia)
G = ultimo_registro(geografia)

maestro = (P
    .merge(L.drop(columns="periodo"), on="id_persona", how="left")
    .merge(C.drop(columns="periodo"), on="id_persona", how="left")
    .merge(G.drop(columns="periodo"), on="id_persona", how="left"))

# Unificacion del score: convive SCORE_ALTO con ALTO para el mismo nivel.
maestro["score"] = (maestro["score_riesgo"].astype(str)
                    .str.replace("^SCORE_", "", regex=True)
                    .replace("nan", np.nan))

print("maestro (una fila por persona):", maestro.shape)

maestro (una fila por persona): (674546, 29)


In [76]:
# Test 1: distancia al polo comercial. Si la distribucion es identica en cada CP, no informa.
dist = maestro.groupby("cp")["distancia_estimada_polo_comercial_km"].agg(["mean", "std"]).round(2)
print("distancia al polo comercial - media por CP: min %.2f / max %.2f  (std ~%.2f en todos)"
      % (dist["mean"].min(), dist["mean"].max(), dist["std"].median()))
print("distribucion global:")
print(pd.cut(maestro["distancia_estimada_polo_comercial_km"],
             bins=np.arange(0, 5.5, 0.5)).value_counts().sort_index().to_string())

distancia al polo comercial - media por CP: min 2.47 / max 2.55  (std ~1.44 en todos)
distribucion global:
distancia_estimada_polo_comercial_km
(0.0, 0.5]    67607
(0.5, 1.0]    67189
(1.0, 1.5]    67201
(1.5, 2.0]    67614
(2.0, 2.5]    67304
(2.5, 3.0]    67563
(3.0, 3.5]    67730
(3.5, 4.0]    67868
(4.0, 4.5]    67555
(4.5, 5.0]    66208


In [77]:
# Test 2: segmento de comportamiento retail contra categoria de ingreso.
# Si el cruce da 25/25/25/25 en todas las filas, las dos variables son independientes
# y el segmento es una etiqueta asignada al azar.
tab = pd.crosstab(maestro["categoria_ingreso"],
                  maestro["segmento_comportamiento_retail"], normalize="index") * 100
display(tab.round(1))

segmento_comportamiento_retail,Frecuente,Habitual,Ocasional,Premium/Heavy User
categoria_ingreso,,,,
ALTO,25.20,25.10,24.60,25.00
BAJO,24.80,25.10,25.10,25.00
MEDIO,25.10,25.10,25.00,24.90
MEDIO-ALTO,25.20,24.80,24.90,25.10
MEDIO-BAJO,25.10,24.80,25.20,24.90


In [78]:
# Contraste: las variables que SI tienen senal muestran estructura clara.
print("score de riesgo por categoria de ingreso (% por fila):")
display((pd.crosstab(maestro["categoria_ingreso"], maestro["score"], normalize="index") * 100).round(1))

print("\nbancarizacion por categoria de ingreso (% por fila):")
display((pd.crosstab(maestro["categoria_ingreso"], maestro["bancarizacion"], normalize="index") * 100).round(1))

score de riesgo por categoria de ingreso (% por fila):


score,ALTO,BAJO,MEDIO,MEDIO_ALTO,MEDIO_BAJO
categoria_ingreso,,,,,
ALTO,67.70,6.60,5.10,16.90,3.70
BAJO,2.20,23.30,30.60,27.90,16.00
MEDIO,36.10,18.30,12.10,25.90,7.60
MEDIO-ALTO,54.20,11.40,7.50,21.70,5.20
MEDIO-BAJO,8.40,15.80,23.90,39.70,12.20



bancarizacion por categoria de ingreso (% por fila):


bancarizacion,HIT,THIN
categoria_ingreso,,
ALTO,99.70,0.30
BAJO,62.40,37.60
MEDIO,92.60,7.40
MEDIO-ALTO,97.90,2.10
MEDIO-BAJO,53.30,46.70


## 7. El problema central: la falta de datos esta organizada por plaza

Este es el hallazgo que condiciona todo el analisis. Los nulos de las tablas de ingresos,
laboral y crediticia **no estan repartidos al azar entre personas**: hay codigos postales
enteros donde una variable esta completa y codigos postales enteros donde no existe.

Consecuencia directa: cualquier indice que combine ingreso, situacion laboral y score
compara plazas que no son comparables, y castiga en silencio a las plazas ciegas.

In [79]:
cobertura = (maestro.groupby("cp")
             .agg(personas=("id_persona", "size"),
                  cob_ingreso=("categoria_ingreso", lambda s: round(s.notna().mean() * 100, 1)),
                  cob_score=("score_riesgo", lambda s: round(s.notna().mean() * 100, 1)),
                  cob_laboral=("es_relacion_dependencia", lambda s: round(s.notna().mean() * 100, 1)),
                  cob_hogar=("cantidad_personas_hogar", lambda s: round(s.notna().mean() * 100, 1)))
             .sort_values("personas", ascending=False))
display(cobertura.head(20))

,personas,cob_ingreso,cob_score,cob_laboral,cob_hogar
cp,,,,,
5000,86711,99.80,100.00,99.80,27.80
2000,69030,99.80,100.00,99.80,28.90
7600,47009,99.80,99.90,99.70,34.30
4000,39299,0.50,0.60,99.70,22.80
1900,36612,0.40,100.00,99.80,28.50
4400,35663,0.50,99.90,99.70,23.00
3000,26059,99.80,100.00,99.70,29.90
1425,24319,0.40,100.00,99.90,26.60
3400,22139,99.80,100.00,0.40,22.30


In [80]:
# Cuantas plazas quedan ciegas en cada bloque y cuanta muestra representan
n_tot = len(maestro)
bloques = {"categoria_ingreso": "ingreso",
           "score_riesgo": "score crediticio",
           "es_relacion_dependencia": "situacion laboral"}

for col, nombre in bloques.items():
    cob = maestro.groupby("cp")[col].apply(lambda s: s.notna().mean())
    ciegos = sorted(cob[cob < 0.05].index.astype(str).tolist())
    afectadas = maestro["cp"].astype(str).isin(ciegos).sum()
    print(f"{nombre:<18} plazas sin dato: {len(ciegos):>2}/47   "
          f"personas afectadas: {afectadas:>7,} ({afectadas/n_tot*100:4.1f}%)")
    print(f"{'':<18} CPs: {ciegos}\n")

ingreso            plazas sin dato:  4/47   personas afectadas: 135,893 (20.1%)
                   CPs: ['1425', '1900', '4000', '4400']

score crediticio   plazas sin dato:  2/47   personas afectadas:  46,396 ( 6.9%)
                   CPs: ['1712', '4000']

situacion laboral  plazas sin dato: 13/47   personas afectadas: 177,193 (26.3%)
                   CPs: ['1405', '1407', '1408', '1686', '1704', '1744', '1824', '1828', '1888', '3400', '3500', '8300', '8400']



In [81]:
# Patron de disponibilidad por plaza: que combinacion de bloques tiene cada CP
disp = maestro.groupby("cp").agg(
    n=("id_persona", "size"),
    ing=("categoria_ingreso", lambda s: s.notna().mean() > .5),
    lab=("es_relacion_dependencia", lambda s: s.notna().mean() > .5),
    sco=("score_riesgo", lambda s: s.notna().mean() > .5))

disp["patron"] = (disp["ing"].map({True: "ING", False: "---"}) + "/"
                + disp["lab"].map({True: "LAB", False: "---"}) + "/"
                + disp["sco"].map({True: "SCO", False: "---"}))

resumen = (disp.groupby("patron")
           .agg(plazas=("n", "size"), personas=("n", "sum"))
           .assign(pct_muestra=lambda d: (d["personas"] / len(maestro) * 100).round(1))
           .sort_values("personas", ascending=False))
display(resumen)

completas = sorted(disp[disp["patron"] == "ING/LAB/SCO"].index.astype(str).tolist())
print(f"Plazas con los tres bloques completos: {len(completas)} de 47")
print(completas)

,plazas,personas,pct_muestra
patron,,,
ING/LAB/SCO,29,354363,52.50
ING/---/SCO,13,177193,26.30
---/LAB/SCO,3,96594,14.30
---/LAB/---,1,39299,5.80
ING/LAB/---,1,7097,1.10


Plazas con los tres bloques completos: 29 de 47
['1406', '1427', '1429', '1430', '1609', '1613', '1615', '1617', '1653', '1665', '1678', '1716', '1722', '1752', '1763', '1852', '1875', '1881', '2000', '2300', '2800', '2804', '3000', '5000', '5300', '5730', '7600', '8000', '9100']


## 8. Foto preliminar de las plazas

Primer dimensionamiento de los CP candidatos. **Advertencia:** las columnas de ingreso,
laboral y score solo son comparables entre plazas que tienen ese bloque disponible
(seccion 7). Los valores cercanos a cero en esas columnas indican ausencia de dato,
no ausencia del atributo.

In [82]:
m = maestro
m["_bueno"] = m["score"].isin(["ALTO", "MEDIO_ALTO"])
m["_ing_alto"] = m["categoria_ingreso"].isin(["ALTO", "MEDIO-ALTO"])
m["_hit"] = m["bancarizacion"].eq("HIT")
m["_demanda"] = m["cantidad_solicitudes_financiamiento_12m"].isin(["ENTRE 1 Y 3", "4 O MÁS"])

plazas = (m.groupby("cp").agg(
        personas=("id_persona", "size"),
        edad_media=("edad", "mean"),
        pct_bancarizado=("_hit", "mean"),
        pct_demanda_12m=("_demanda", "mean"),
        lineas_activas=("cantidad_lineas_credito_activas_12m", "mean"),
        pct_score_bueno=("_bueno", "mean"),
        pct_ingreso_alto=("_ing_alto", "mean"))
      .sort_values("personas", ascending=False))

for col in ["pct_bancarizado", "pct_demanda_12m", "pct_score_bueno", "pct_ingreso_alto"]:
    plazas[col] = (plazas[col] * 100).round(1)
plazas["edad_media"] = plazas["edad_media"].round(1)
plazas["lineas_activas"] = plazas["lineas_activas"].round(2)
plazas["%_muestra"] = (plazas["personas"] / plazas["personas"].sum() * 100).round(2)

display(plazas.head(20))
print("total plazas:", len(plazas), "| total personas:", f"{plazas['personas'].sum():,}")

,personas,edad_media,pct_bancarizado,pct_demanda_12m,lineas_activas,pct_score_bueno,pct_ingreso_alto,%_muestra
cp,,,,,,,,
5000,86711,45.40,75.50,45.20,2.21,53.50,19.00,12.85
2000,69030,46.00,73.80,42.10,2.35,61.30,20.60,10.23
7600,47009,46.60,68.10,40.00,1.94,60.50,17.40,6.97
4000,39299,44.90,75.00,47.20,2.57,0.30,0.10,5.83
1900,36612,47.30,73.00,38.00,1.96,68.50,0.10,5.43
4400,35663,44.10,72.10,42.30,2.50,46.00,0.10,5.29
3000,26059,45.20,75.50,43.80,2.55,53.90,18.10,3.86
1425,24319,50.30,70.90,37.80,2.40,85.20,0.10,3.61
3400,22139,44.40,76.40,45.00,2.17,44.00,12.50,3.28


total plazas: 47 | total personas: 674,546


In [83]:
# Variables disponibles en las 47 plazas, aptas para un indice comparable
universales = ["personas", "edad_media", "pct_bancarizado", "pct_demanda_12m", "lineas_activas"]
print("Dispersion entre plazas de las variables sin huecos geograficos:\n")
for col in universales[1:]:
    print(f"{col:<20} min={plazas[col].min():>7}   max={plazas[col].max():>7}   "
          f"mediana={plazas[col].median():>7}")

Dispersion entre plazas de las variables sin huecos geograficos:

edad_media           min=   36.1   max=   50.3   mediana=   44.4
pct_bancarizado      min=   63.1   max=   82.6   mediana=   73.0
pct_demanda_12m      min=   35.6   max=   50.4   mediana=   42.3
lineas_activas       min=   1.48   max=   2.81   mediana=   2.18


## 9. Sintesis y decisiones pendientes

Resumen de lo detectado, para pegar en el log de decisiones del equipo. Cada linea necesita
una decision explicita antes de construir el indice de atractivo de plazas.

## 10. Hallazgos consolidados

Tabla ejecutable con los problemas detectados y su tratamiento pendiente.


In [84]:
hallazgos = [
    ("estructura",  "El universo son los CP presentes en geografia_hogar: ese es el conjunto de plazas candidatas."),
    ("estructura",  "La mayoria de las personas aparece en un solo periodo: no es un panel, es un corte transversal apilado."),
    ("critico",     "score_riesgo tiene dos codificaciones (SCORE_X y X). Unificar antes de cualquier agregacion."),
    ("critico",     "informacion_crediticia tiene filas sin periodo: definir si se imputan o se descartan."),
    ("duplicados",  "Las tres tablas de atributos tienen duplicados por (id_persona, periodo), casi todos identicos."),
    ("incoherencia","lineas_credito_activas: 24m y 36m son la misma columna, y hay filas con 12m > 24m."),
    ("incoherencia","Hay personas con mas de una situacion laboral TRUE en simultaneo."),
    ("outliers",    "Edades por encima de 100 anios y cantidades de lineas de credito extremas."),
    ("nulos",       "El bloque de variables de hogar falta en la mayoria de los registros: sirve para perfilar, no para dimensionar."),
    ("varianza",    "beneficiario_plan_social_12m y score_fraude casi no varian: no sirven como drivers del indice."),
]
display(pd.DataFrame(hallazgos, columns=["tipo", "hallazgo"]))

,tipo,hallazgo
0,estructura,El universo son los CP presentes en geografia_...
1,estructura,La mayoria de las personas aparece en un solo ...
2,critico,score_riesgo tiene dos codificaciones (SCORE_X...
3,critico,informacion_crediticia tiene filas sin periodo...
4,duplicados,Las tres tablas de atributos tienen duplicados...
5,incoherencia,lineas_credito_activas: 24m y 36m son la misma...
6,incoherencia,Hay personas con mas de una situacion laboral ...
7,outliers,Edades por encima de 100 anios y cantidades de...
8,nulos,El bloque de variables de hogar falta en la ma...
9,varianza,beneficiario_plan_social_12m y score_fraude ca...


### Proximo paso

Con este diagnostico cerrado, el siguiente notebook aplica las decisiones de limpieza y
produce las dos salidas que consume el resto del equipo:

- `maestro_personas.csv` — una fila por persona, con el registro mas reciente de cada tabla.
- `plazas.csv` — una fila por CP, con todas las metricas agregadas del mercado.

A partir de ese momento nadie vuelve a leer los `.parquet` crudos.